In [2]:
# Import required libraries
import numpy as np
import pandas as pd
import plotly.graph_objects as go

In [3]:
# Example: Load your dataframe (replace with your actual data loading method)
data = {
    'description': ['Base', 'Scene A', 'Scene B', 'Scene C', 'Back to Base'],
    'timestamp': [0, 10, 20, 30, 40],
    'level': [0, 1, 2, 3, 0]
}
df = pd.DataFrame(data)
df

,description,timestamp,level
0,Base,0,0
1,Scene A,10,1
2,Scene B,20,2
3,Scene C,30,3
4,Back to Base,40,0


In [4]:
descriptions=[
    "Interview", "Base Scene", "Tour of the Palace","At the Pub","KwikFit Kult",
    "Base Scene","Pyramid Scheme","Pyramid Scheme - 2nd Sale",
    "Base Scene","Jerry's Dry Cleaners","Laundry Drop Off","Auntie Wants That Hat","Sacrificed for a Hat",
    "Base Scene", "Hatters", "Police Enquiry",
    "Base Scene","Pack Your Bags Meghan",
    "Base Scene",
    "End"
    ]
timestamps=[
    "00:00", "01:50","03:36","04:39","05:04",  
    "5:50","06:26","06:59",
    "7:51","8:39","9:06","9:38","10:07",
    "10:18", "10:38","11:36",
    "12:27","14:06",
    "14:38",
    "15:00"
    ]
levels=[
    0, 1, 2, 3, 4,
1,2,3,
1,2,3,4,5,
1,2,3,
1,2,
1,
1]

#convert list of strings of format "MM:SS" to list of integers in seconds
def convert_to_seconds(timestamps):
    seconds = []
    for t in timestamps:
        m, s = map(int, t.split(':'))
        total_seconds = m * 60 + s
        seconds.append(total_seconds)
    return seconds

timestamps = convert_to_seconds(timestamps)

data = {
    'description': descriptions,
    'timestamp': timestamps,
    'level': levels
}

df = pd.DataFrame(data)
df

,description,timestamp,level
0,Interview,0,0
1,Base Scene,110,1
2,Tour of the Palace,216,2
3,At the Pub,279,3
4,KwikFit Kult,304,4
5,Base Scene,350,1
6,Pyramid Scheme,386,2
7,Pyramid Scheme - 2nd Sale,419,3
8,Base Scene,471,1
9,Jerry's Dry Cleaners,519,2


In [5]:
len(descriptions), len(timestamps), len(levels)

(20, 20, 20)

In [7]:
# Plot straight lines for constant levels and arcs for transitions, colored by level (no markers), annotated with description and pastel colors, with black text on top of the line
import plotly.colors as pc
from matplotlib import cm
from matplotlib.colors import Normalize, to_hex

# Use a pastel color wheel from plotly
pastel_colors = pc.qualitative.Pastel1
levels = df['level'].unique()
level_to_color = {level: pastel_colors[i % len(pastel_colors)] for i, level in enumerate(sorted(levels))}
def get_level_color(level):
    return level_to_color[level]

def bezier_arc(x0, y0, x1, y1, arc_height=0.5, steps=30):
    xm = (x0 + x1) / 2
    ym = (y0 + y1) / 2 + arc_height * (1 if y1 > y0 else -1)
    t = np.linspace(0, 1, steps)
    x = (1-t)**2 * x0 + 2*(1-t)*t * xm + t**2 * x1
    y = (1-t)**2 * y0 + 2*(1-t)*t * ym + t**2 * y1
    return x, y

# Convert seconds to MM:SS for x-axis labels
def seconds_to_mmss(seconds):
    m = int(seconds // 60)
    s = int(seconds % 60)
    return f"{m:02d}:{s:02d}"

df['timestamp_mmss'] = [seconds_to_mmss(ts) for ts in df['timestamp']]

fig = go.Figure()
for i in range(len(df)-1):
    x0, y0 = df['timestamp'][i], df['level'][i]
    x1, y1 = df['timestamp'][i+1], df['level'][i+1]
    x0_mmss, x1_mmss = df['timestamp_mmss'][i], df['timestamp_mmss'][i+1]
    color = get_level_color(y0)
    desc = df['description'][i+1]  # annotate with next scene's description
    if y0 == y1:
        fig.add_trace(go.Scatter(x=[x0_mmss, x1_mmss], y=[y0, y1], mode='lines+text', line=dict(color=color, width=4), text=[None, desc], textposition='top center', textfont=dict(size=14, color='black'), showlegend=False))
    else:
        arc_start = x1 - (x1 - x0) * 0.2
        arc_start_mmss = seconds_to_mmss(arc_start)
        fig.add_trace(go.Scatter(x=[x0_mmss, arc_start_mmss], y=[y0, y0], mode='lines', line=dict(color=color, width=4), showlegend=False))
        arc_x, arc_y = bezier_arc(arc_start, y0, x1, y1, arc_height=0.7)
        arc_x_mmss = [seconds_to_mmss(x) for x in arc_x]
        # Only annotate at the end of the arc
        arc_text = [None]*(len(arc_x_mmss)-1) + [desc]
        fig.add_trace(go.Scatter(x=arc_x_mmss, y=arc_y, mode='lines+text', line=dict(color=color, width=4), text=arc_text, textposition='bottom center', textfont=dict(size=14, color='black'), showlegend=False))

fig.update_layout(title='Scamps - Cage Match Qualifier - 26/09/25',
                  xaxis_title='Time (MM:SS)',
                  yaxis_title='Level',
                  yaxis=dict(dtick=1),
                  showlegend=False,
                  width=1800,
                  height=700)
fig.show()

In [15]:
import plotly.graph_objects as go
import plotly.colors as pc
import numpy as np

# === colors / helpers ===
pastel_colors = pc.qualitative.Pastel1
levels = df['level'].unique()
level_to_color = {level: pastel_colors[i % len(pastel_colors)] for i, level in enumerate(sorted(levels))}
def get_level_color(level):
    return level_to_color[level]

def bezier_arc(x0, y0, x1, y1, arc_height=0.5, steps=40):
    xm = (x0 + x1) / 2
    ym = (y0 + y1) / 2 + arc_height * (1 if y1 > y0 else -1)
    t = np.linspace(0, 1, steps)
    x = (1-t)**2 * x0 + 2*(1-t)*t * xm + t**2 * x1
    y = (1-t)**2 * y0 + 2*(1-t)*t * ym + t**2 * y1
    return x, y

def seconds_to_mmss(seconds):
    total = int(round(seconds))
    m = total // 60
    s = total % 60
    return f"{m:02d}:{s:02d}"

# create MM:SS column (keeps same approach you used)
df['timestamp_mmss'] = [seconds_to_mmss(ts) for ts in df['timestamp']]

# === build figure: add ALL line traces first, collect annotations ===
fig = go.Figure()
annotations = []

# --- add annotation for the very first row ---
annotations.append(dict(
    x=df['timestamp_mmss'].iloc[0],
    y=df['level'].iloc[0],
    text=df['description'].iloc[0]
))

# --- now loop through transitions as before ---
for i in range(len(df)-1):
    x0, y0 = df['timestamp'][i], df['level'][i]
    x1, y1 = df['timestamp'][i+1], df['level'][i+1]
    x0_mmss, x1_mmss = df['timestamp_mmss'][i], df['timestamp_mmss'][i+1]
    color = get_level_color(y0)
    desc = df['description'][i+1]  # next scene description


    if y0 == y1:
        # straight line segment
        fig.add_trace(go.Scatter(
            x=[x0_mmss, x1_mmss],
            y=[y0, y1],
            mode='lines',
            line=dict(color=color, width=4),
            hoverinfo='none',
            showlegend=False
        ))
        # queue an annotation at the segment endpoint
        annotations.append(dict(x=x1_mmss, y=y1, text=desc))

    else:
        # straight run-up segment before the arc
        arc_start = x1 - (x1 - x0) * 0.2
        arc_start_mmss = seconds_to_mmss(arc_start)
        fig.add_trace(go.Scatter(
            x=[x0_mmss, arc_start_mmss],
            y=[y0, y0],
            mode='lines',
            line=dict(color=color, width=4),
            hoverinfo='none',
            showlegend=False
        ))

        # arc curve (bezier)
        arc_x, arc_y = bezier_arc(arc_start, y0, x1, y1, arc_height=0.7, steps=40)
        arc_x_mmss = [seconds_to_mmss(x) for x in arc_x]
        fig.add_trace(go.Scatter(
            x=arc_x_mmss,
            y=arc_y,
            mode='lines',
            line=dict(color=color, width=4),
            hoverinfo='none',
            showlegend=False
        ))

        # queue an annotation at the arc endpoint
        annotations.append(dict(x=x1_mmss, y=y1, text=desc))

# === add annotations AFTER all lines (annotations sit on top) ===
for ann in annotations:
    fig.add_annotation(
        x=ann['x'],
        y=ann['y'],
        xref='x',
        yref='y',
        text=ann['text'],
        showarrow=False,
        font=dict(color='black', size=14),
        bgcolor='rgba(255,255,255,0.85)',  # soft white box so text is readable
        bordercolor='rgba(0,0,0,0)',
        align='center',
        yshift=10   # push the text visually above the curve (in pixels)
    )

fig.update_layout(
    title='Scamps - Cage Match Qualifier - 26/09/25',
    xaxis_title='Time (MM:SS)',
    yaxis_title='Level',
    yaxis=dict(dtick=1),
    showlegend=False,
    width=1800,
    height=700
)

fig.show()
